# The Boundary Lever

## 03 — Results

The predictions committed in `01_predictions.ipynb`, scored against data not used
to build the claim.


In [1]:
import sys, os, math, itertools
sys.path.insert(0, os.path.abspath('/home/rendier/Projects/ThePlace'))
import numpy as np
from ValaQuenta.modules.box_kite.maths import cd_multiplication_table
np.set_printoptions(linewidth=120)


In [2]:
def cd(level):
    """Cayley-Dickson table and dimension at a doubling level."""
    return cd_multiplication_table(level)

def basis_ops(tab, dim):
    """P[k] = the (dim x dim) matrix of LEFT multiplication by e_k.

    Precomputing these once turns every L_a for a basis-pair divisor into a
    single matrix add, instead of a dim^2 Python loop per candidate. Without
    this the dim=128 sweep is hours; with it, seconds."""
    P = np.zeros((dim, dim, dim))
    for i in range(dim):
        for j in range(dim):
            s, k = tab[(i, j)]        # box_kite returns (SIGN, INDEX)
            P[i, k, j] = s
    return P

def diagonal_L(P, i, j, sgn):
    """L_a for a = (e_i + sgn*e_j)/sqrt(2)."""
    return (P[i] + sgn * P[j]) / math.sqrt(2.0)

def zd_census(level, tol=1e-10, verbose=False):
    """Every (i<j, sign) whose diagonal is a zero divisor, with its nullity."""
    tab, dim = cd(level); H = dim // 2
    P = basis_ops(tab, dim)
    rows = []
    for i in range(dim):
        for j in range(i + 1, dim):
            for sgn in (1.0, -1.0):
                s = np.linalg.svd(diagonal_L(P, i, j, sgn), compute_uv=False)
                n = int((s < tol).sum())
                if n:
                    loc = 'lower' if j < H else ('upper' if i >= H else 'cross')
                    rows.append({'i': i, 'j': j, 'sign': int(sgn), 'nullity': n, 'loc': loc})
    return dim, H, rows

def orphans_from(dim, rows):
    used = {r['i'] for r in rows} | {r['j'] for r in rows}
    return [k for k in range(dim) if k not in used]

def mirror_test(level):
    """Is e_(i+H) e_(j+H) the lower product with the order REVERSED?"""
    tab, dim = cd(level); H = dim // 2
    idx = [k for k in range(1, H)]
    rev = fwd = tot = 0
    for i in idx:
        for j in idx:
            if i == j: continue
            up = tab[(i + H, j + H)]
            if up == tab[(j, i)]: rev += 1
            if up == tab[(i, j)]: fwd += 1
            tot += 1
    esc = sum(1 for i in range(H) for j in range(H) if tab[(i + H, j + H)][1] >= H)
    return {'dim': dim, 'H': H, 'reversed': rev, 'preserved': fwd, 'pairs': tot,
            'upper_escapes_lower': esc, 'upper_pairs': H * H}


In [3]:
import json, pathlib
PRED = json.loads(pathlib.Path('predictions.json').read_text())
SCORE = {}
for k, v in PRED.items():
    print(f"{k}  {v['what']:<40} expect {v['expect']}")


P1  orphans at dim=128                       expect [0, 64]
P2  mirror order-reversing at dim=128        expect {'reversed': 3782, 'preserved': 0, 'pairs': 3782}
P3  upper*upper escapes lower at dim=128     expect 0
P4  ZD census at dim=64                      expect {'lower': 588, 'cross': 2940, 'upper': 588, 'total': 4116}
P5  nullity classes at dim=64                expect {'non_crossing': [16], 'crossing': [8, 24]}


## P4 / P5 — the `dim = 64` census

The extrapolation from a single step. Risky by construction.


In [4]:
dim64, H64, rows64 = zd_census(6)
locs = ['lower', 'cross', 'upper']
tot64 = {l: sum(1 for r in rows64 if r['loc'] == l) for l in locs}
tot64['total'] = len(rows64)
print('P4  measured:', tot64)
print('P4  predicted:', PRED['P4']['expect'])
SCORE['P4'] = (tot64 == PRED['P4']['expect'])
print('P4  ->', 'CONFIRMED' if SCORE['P4'] else 'FALSIFIED')


P4  measured: {'lower': 588, 'cross': 1860, 'upper': 588, 'total': 3036}
P4  predicted: {'lower': 588, 'cross': 2940, 'upper': 588, 'total': 4116}
P4  -> FALSIFIED


In [5]:
non_cross = sorted({r['nullity'] for r in rows64 if r['loc'] in ('lower', 'upper')})
crossing  = sorted({r['nullity'] for r in rows64 if r['loc'] == 'cross'})
print(f"P5  measured : non-crossing {non_cross}   crossing {crossing}")
print(f"P5  predicted: non-crossing {PRED['P5']['expect']['non_crossing']}   crossing {PRED['P5']['expect']['crossing']}")
SCORE['P5'] = (non_cross == PRED['P5']['expect']['non_crossing']
               and crossing == PRED['P5']['expect']['crossing'])
print('P5  ->', 'CONFIRMED' if SCORE['P5'] else 'FALSIFIED')
print()
print('separation still perfect (no nullity in both classes):',
      set(non_cross).isdisjoint(set(crossing)))


P5  measured : non-crossing [8, 16, 24]   crossing [4, 12, 20, 28]
P5  predicted: non-crossing [16]   crossing [8, 24]
P5  -> FALSIFIED

separation still perfect (no nullity in both classes): True


## P2 / P3 — the mirror at `dim = 128`


In [6]:
m128 = mirror_test(7)
print('P2  measured :', {k: m128[k] for k in ('reversed', 'preserved', 'pairs')})
print('P2  predicted:', PRED['P2']['expect'])
SCORE['P2'] = ({k: m128[k] for k in ('reversed', 'preserved', 'pairs')} == PRED['P2']['expect'])
print('P2  ->', 'CONFIRMED' if SCORE['P2'] else 'FALSIFIED')
print()
print(f"P3  measured : {m128['upper_escapes_lower']} / {m128['upper_pairs']}")
print(f"P3  predicted: {PRED['P3']['expect']}")
SCORE['P3'] = (m128['upper_escapes_lower'] == PRED['P3']['expect'])
print('P3  ->', 'CONFIRMED' if SCORE['P3'] else 'FALSIFIED')


P2  measured : {'reversed': 3906, 'preserved': 0, 'pairs': 3906}
P2  predicted: {'reversed': 3782, 'preserved': 0, 'pairs': 3782}
P2  -> FALSIFIED

P3  measured : 0 / 4096
P3  predicted: 0
P3  -> CONFIRMED


## P1 — THE FALSIFIER

> Compute the orphans at `dim = 128`. If they are not exactly `[0, 64]`,
> **the claim is dead.**

`(128 x 127)/2 x 2 = 16256` candidate diagonals, each a `128 x 128` SVD. The
precomputed basis operators make this tractable; expect a minute or two.


In [7]:
import time
t0 = time.time()
dim128, H128, rows128 = zd_census(7)
orp128 = orphans_from(dim128, rows128)
print(f'({time.time()-t0:.0f}s, {len(rows128)} ZD diagonals found)')
print()
print('P1  measured :', orp128)
print('P1  predicted:', PRED['P1']['expect'])
SCORE['P1'] = (orp128 == PRED['P1']['expect'])
print('P1  ->', 'CONFIRMED' if SCORE['P1'] else 'FALSIFIED')


(28s, 13884 ZD diagonals found)

P1  measured : [0, 64]
P1  predicted: [0, 64]
P1  -> CONFIRMED


## Scorecard


In [8]:
print(f"{'#':<5}{'prediction':<44}{'verdict':<12}")
print('-' * 61)
for k in sorted(SCORE):
    print(f"{k:<5}{PRED[k]['what']:<44}{'CONFIRMED' if SCORE[k] else 'FALSIFIED':<12}")
print('-' * 61)
print(f"{sum(SCORE.values())}/{len(SCORE)} confirmed")
print()
print('THE CLAIM (stands or falls on P1):',
      'STANDS' if SCORE['P1'] else 'FALSIFIED')


#    prediction                                  verdict     
-------------------------------------------------------------
P1   orphans at dim=128                          CONFIRMED   
P2   mirror order-reversing at dim=128           FALSIFIED   
P3   upper*upper escapes lower at dim=128        CONFIRMED   
P4   ZD census at dim=64                         FALSIFIED   
P5   nullity classes at dim=64                   FALSIFIED   
-------------------------------------------------------------
2/5 confirmed

THE CLAIM (stands or falls on P1): STANDS


---

## Reading the failures honestly

Three of five predictions failed. They failed in three different ways and the
differences matter more than the count.

### P2 — failed on my arithmetic, not on the algebra

Predicted `3782` reversed pairs; measured `3906`. But measured **`preserved: 0`**
and `reversed == pairs` exactly — the mirror property held perfectly.

The error was in deriving the pair count: `H = 64` gives `idx = range(1, 64)`,
so `63 x 62 = 3906`, not `62 x 61 = 3782`. **A bookkeeping mistake in the
prediction, not a discovery about the boundary.** Recorded as FALSIFIED because
that is what the committed number says, and predictions are not re-fitted after
the fact. The substantive content — order-reversal is total, exception-free —
is confirmed at `dim = 128`.

### P4 — the extrapolation is simply dead

`lower = upper = 588` held. **Crossing did not:** `1860` measured against `2940`
predicted, total `3036` against `4116`.

The census runs `84 -> 588 -> 3036`, ratios `7` then `5.16`. Not geometric. The
`x7` rule was read off a single step and broke at the next one — exactly the risk
flagged when it was committed. No rescue is available and none is attempted.

### P5 — failed, and the failure is better than the prediction was

⚠ **What follows is POST HOC.** It was found by looking at data that had already
falsified P5. It is an observation, not a result, until it survives a level it was
not derived from.


In [9]:
# The measured nullity classes, as multiples of 4.
for lv, label in ((5, 'dim  32'), (6, 'dim  64')):
    _, _, rws = zd_census(lv)
    nc = sorted({r['nullity'] for r in rws if r['loc'] in ('lower', 'upper')})
    cr = sorted({r['nullity'] for r in rws if r['loc'] == 'cross'})
    print(f"{label}   non-crossing {str(nc):<16} /4 = {[n//4 for n in nc]}")
    print(f"{'':7}   crossing     {str(cr):<16} /4 = {[n//4 for n in cr]}")
    print(f"{'':7}   non-crossing all EVEN multiples of 4: {all((n//4) % 2 == 0 for n in nc)}")
    print(f"{'':7}   crossing     all ODD  multiples of 4: {all((n//4) % 2 == 1 for n in cr)}")
    print()


dim  32   non-crossing [8]              /4 = [2]
          crossing     [4, 12]          /4 = [1, 3]
          non-crossing all EVEN multiples of 4: True
          crossing     all ODD  multiples of 4: True



dim  64   non-crossing [8, 16, 24]      /4 = [2, 4, 6]
          crossing     [4, 12, 20, 28]  /4 = [1, 3, 5, 7]
          non-crossing all EVEN multiples of 4: True
          crossing     all ODD  multiples of 4: True



### The observation

> **A zero divisor confined to one half has nullity equal to an EVEN multiple of 4.**
> **A zero divisor spanning the boundary has nullity equal to an ODD multiple of 4.**

Holds at `dim = 32` and `dim = 64`, with no exceptions in either direction, and it
subsumes the `dim = 32` observation that first suggested P5.

P5 predicted the wrong thing because it guessed a *value* (`dim/4`) where the real
structure is a *parity*. The separation itself — the thing that made P5 seem worth
predicting — is intact and sharper than before.

**Pre-registered now, for the next level and before it is computed:**

> **P6.** At `dim = 256`, non-crossing nullities are even multiples of 4 and
> crossing nullities are odd multiples of 4, with no index in both classes.

P6 is committed here and deliberately left untested in this paper. Testing a
post-hoc pattern on the data that produced it would repeat the error P4 and P5
were designed to catch.


### What survives

| | |
|---|---|
| **The claim** | **STANDS.** P1, its sole falsifier, confirmed at `dim = 128`. |
| Mirror is chiral and total | confirmed at 128 (P2's substance; its count was mis-derived) |
| Upper half never closes | confirmed at 128 (P3) |
| ZD census extrapolation | **dead** (P4) |
| Nullity = `dim/4` rule | **dead** (P5), replaced by a parity law, untested |

The claim was written to rest on P1 alone, and that is why it survives three
falsified predictions without any of them being reinterpreted.


## The lever, stated once the numbers are in

The boundary is a **chiral mirror**. Its reflection has **exactly two fixed points**,
and those two points are precisely the indices that pair with nothing — the identity
and the newest generator. They are in no zero-divisor plane, they carry no force,
and everything else balances across them in equal number.

**That is a fulcrum.** A lever's pivot does no work; it is the thing that does not
move while the rest balances. The null subspace was already described in 0_RB as
*gravity, present as absence* — dimensions that are indexed, participate, and return
zero. Here that description acquires a mechanical role rather than a poetic one.

And because the mirror is **chiral**, the levels do not stack into a fan of parallel
reflections. Two reflections compose to a rotation, and successive boundaries sit at
`dim/2` — constant step in `log2`, constant pitch `ln 2`. Rotation plus constant
logarithmic advance is a **spiral**, which is the Archimedes screw of Phase 24
arriving from the algebra rather than from the primes.

Chirality is not a detail of the mirror. It is what makes the tower wind.


## Open, and honestly bounded

| item | status |
|---|---|
| `dim = 256` and beyond | untested; the claim asserts it |
| a **proof** rather than a sweep | none — every level here is verified by enumeration, not derived |
| why `nullity` separates crossing from non-crossing | observed exactly, unexplained |
| the spiral pitch as a *derived* quantity | `ln 2` is read off the doubling, not derived from the algebra |
| any claim about observers, meaning, or physics | **out of scope** — see `00_vision` |

⚠ The largest gap is the second row. A sweep that holds at four levels is evidence,
not a theorem. The claim is stated for *every* level and is verified at four of them;
a derivation from the doubling rule would settle it and does not yet exist.

---

*Engine: `ValaQuenta/modules/angular_rank/`, `ValaQuenta/modules/box_kite/`.
Wiki: written last, per protocol.*
